# Testing

## Behaviour you can trust

From the outset of this course, I have emphasised that software quality is not just style or elegance: behaviour must be **demonstrably validated through tests**. A useful guiding principle is: **"don't believe it unless you can run it".**

In practice, every important assumption should be testable, every transformation checkable, and every result reproducible. Tests are not merely for debugging; they are how we build confidence that code does what we think it does&mdash;especially in scientific projects, where errors can propagate into analyses, publications, and downstream decisions.

## Types of tests

- **Logic tests**: Check core functions and logical workflows with controlled inputs, verifying deterministic, expected outputs.

- **User-case tests**: Check representative, domain-relevant workflows from a user's perspective, verifying that outputs are scientifically meaningful in realistic scenarios.

In modular software, the distinction between **Logic tests** and **User-case tests** is crucial. Logic tests ask whether code is internally consistent; user-case tests ask whether that consistency produces trusted scientific outcomes. The latter does not follow automatically from the former, because scientific validity also depends on assumptions, model structure, and domain context. It is often harder to test objectively, since correctness at this level can be context-dependent and contested.

In mainstream terminology, these categories map naturally to **unit tests** and **integration tests**: isolate internal logic, then validate behaviour across connected components.

- **Unit tests** target a small unit of code (usually a single function, method, or class) in isolation. They should be fast, deterministic, and precise, and align closely with **Logic tests**.

- **Integration tests** check whether multiple components work correctly together (for example: parser + model + solver + output step). They align closely with **User-case tests**, because they exercise more realistic workflows where assumptions and interfaces interact.

A practical strategy is to use both: many unit tests for fast logical feedback, and fewer high-value integration tests for end-to-end scientific confidence.

## When tests get written

In scientific software, tests are often an afterthought. A common trajectory is: prototype in a notebook, tidy and share, then add one or two tests much later (if at all), often near submission or first release. This is understandable. Scientific software frequently starts as exploratory code written to investigate questions whose answers are not yet known; at that stage, "correct" is often still emerging.

The problem appears when prototypes quietly become production code. Scripts that were never meant to be reused become dependencies; functions that merely "seemed to work" become load-bearing parts of analysis pipelines. Each untested assumption, undocumented shortcut, or poorly understood function accumulates **technical debt**&mdash;future obligations usually paid at the worst possible moment. At that point, missing tests are no longer a pragmatic trade-off; they are interest on that debt.

## Test-driven development

**Test-driven development (TDD)** is the end-member alternative to writing tests late: you write a test *before* you implement the function. The workflow is a short loop:

1. **Write a failing test** for a small, well-defined behaviour.
2. **Write the minimal code** needed to make it pass.
3. **Refactor** while keeping tests green.

This _red&ndash;green&ndash;refactor_ cycle keeps tests integral to design. The test becomes a concrete specification of intent, and each function gains a behavioural contract by construction.

In practice, TDD is rarely the natural starting point for new scientific code, because early-stage research often lacks stable expected outputs.

It becomes especially valuable during **refactoring**. When you restructure code with already trusted behaviour, writing tests first provides the "safety net" discussed in the refactoring workflow: it forces clear behavioural intent before edits and ensures the new structure preserves existing behaviour. Even without adopting TDD wholesale, using its principles improves test quality and intentionality.

TDD is also one of the most effective ways to work with AI-assisted development tools. A failing test gives an AI agent an executable definition of success. The agent can propose a change, run tests, inspect failures, and iterate&mdash;without requiring manual review of every intermediate attempt. Existing passing tests provide a hard regression constraint: plausible-looking changes that break prior behaviour are rejected automatically. That combination&mdash;clear target, automated feedback, and regression guard&mdash;makes TDD a highly productive interface between human intent and AI-generated code. Passing tests raise confidence, but they only verify what is specified; missing cases still require human judgment.

## Python example

The details of constructing a test suite vary by language, but the underlying workflow is the same. For illustration, we will use a minimal Python example with one function and one unit test.

Notice the structure: the test function name begins with `test_`, and the `assert` statement expresses expected behaviour in a single, explicit check. This is the core pattern you will reuse across larger test suites.

In [5]:
def celsius_to_kelvin(temp_c: float) -> float:
    """Convert temperature from degrees Celsius to Kelvin."""
    return temp_c + 273.15


def test_celsius_to_kelvin_freezing_point() -> None:
    # 0 degC should map exactly to 273.15 K.
    result = celsius_to_kelvin(0.0)
    assert result == 273.15

### Floating-point comparisons

The test above compares floating-point values with `==`. This works in that specific case because `0.0 + 273.15` happens to be represented exactly in IEEE 754 arithmetic. In general, floating-point operations introduce small rounding errors, and `==` comparisons will fail on values that are correct to any practical precision.

The standard solution is `pytest.approx`, which checks equality within a small relative tolerance (default 1&times;10<sup>&minus;6</sup>). This is the correct approach for almost all numerical comparisons in scientific code and one of the most common mistakes new test writers make.

In [6]:
import pytest


def test_celsius_to_kelvin_boiling_point() -> None:
    # pytest.approx checks within a relative tolerance of 1e-6 by default —
    # the correct approach for floating-point comparisons.
    assert celsius_to_kelvin(100.0) == pytest.approx(373.15)


def test_celsius_to_kelvin_absolute_zero() -> None:
    # Absolute zero: the minimum physically meaningful result.
    assert celsius_to_kelvin(-273.15) == pytest.approx(0.0)

In the example above, the function and its test share the same file, which is convenient for illustration, but not representative of real projects. In practice, tests typically live in a dedicated `tests/` directory or sub-package, kept separate from the main source code. A typical project layout looks like this:

```
my_package/
├── __init__.py
└── physics.py
tests/
├── conftest.py
└── test_physics.py
```

This separation has two practical benefits:

1. The test suite does not get bundled into the installed package, which keeps the distribution lean.
2. The boundary between production code and test code remains explicit and easy to navigate.

For Python projects, [pytest](https://docs.pytest.org/) is the most widely used testing framework. It discovers test functions automatically (any function prefixed with `test_`), produces readable failure output, and supports fixtures for managing shared setup and teardown. The standard library also includes [unittest](https://docs.python.org/3/library/unittest.html), which is more verbose but requires no additional dependencies. Both integrate cleanly with VS Code: discovered tests appear in the Testing panel, where you can run individual tests, inspect failures, and set breakpoints without leaving the editor.

To run your test suite from the command line:

```bash
pytest tests/ -v
```

The `-v` flag produces verbose output, listing each test by name and marking it `PASSED` or `FAILED`. When a test fails, `pytest` prints the assertion that failed and the values on both sides.

### Fixtures

Imagine you are writing ten tests that all need the same sample dataset. Rather than creating that dataset at the top of every test function, a `pytest` **fixture** lets you define it once and have pytest supply it automatically to any test that requests it. Think of it as a shared preparation step that runs before a test&mdash;similar to laying out tools before starting an experiment, rather than fetching them mid-way through every time.

Fixtures are defined with the `@pytest.fixture` decorator. Any test function that includes the fixture's name as a parameter will receive its return value automatically&mdash;no import required. Fixtures shared across multiple test files belong in `conftest.py`, which pytest discovers automatically in the `tests/` directory.

In [7]:
import pytest


# In a real project, this fixture would live in tests/conftest.py so it is
# available to all test files automatically. It is defined here for
# demonstration purposes only.
@pytest.fixture
def sample_temperatures() -> list[float]:
    """Temperatures (°C) covering key physical landmarks."""
    return [-273.15, 0.0, 37.0, 100.0]


def test_kelvin_always_non_negative(sample_temperatures: list[float]) -> None:
    assert all(celsius_to_kelvin(t) >= 0.0 for t in sample_temperatures)

By default, a fixture runs fresh for each test function. This is safe but can be slow if the setup is expensive. `pytest` lets you control this with a `scope` parameter: setting `scope="session"` means the fixture runs once for the entire test run and its result is reused across every test that requests it. This is particularly useful in scientific code where loading a large reference dataset or initialising a complex model takes significant time.

## Striking a balance

As with many subtleties of software development, you must choose the right test granularity for your project. Too many tests can create overlap and redundancy, and can also increase maintenance effort as the codebase evolves. Too few tests, however, leaves real risk of breakage&mdash;especially during refactoring or feature development. The goal is not maximum test count, but a balanced suite that is fast, meaningful, and maintainable.

AI tools can meaningfully accelerate the process of writing tests, particularly where there are clear, repetitive patterns. If you have a function that handles multiple input types, edge cases, or boundary conditions, an AI agent can systematically generate test cases covering that parameter space far faster than writing each by hand. It can also scan existing code and suggest tests for untested functions, helping to close gaps in coverage that would otherwise be easy to miss.

That said, AI-generated tests require the same critical scrutiny as AI-generated code. An agent will write tests that reflect its interpretation of what the function *does*, not necessarily what it *should* do. If the implementation contains a subtle error, the test may encode that error rather than catch it. Human review remains essential: the goal is to use AI to reduce the mechanical burden of test writing, while retaining ownership of what is actually being verified.

This is especially true for user-case tests. Catching a logic error in a conversion function is a relatively mechanical task; deciding whether a model output is scientifically plausible, whether a result is within a physically meaningful range, or whether a workflow faithfully reflects domain assumptions is not. That judgement cannot be delegated to an AI tool, or to a test framework. It belongs to the scientist, and it is precisely where domain expertise becomes irreplaceable.

## Continuous integration

A test suite only provides value if it is actually run. Locally, that means running `pytest` before committing changes. At the project level, it means running tests automatically on every push. **Continuous integration (CI)** services such as [GitHub Actions](https://docs.github.com/en/actions) make this straightforward: a short configuration file in your repository instructs the CI server to install dependencies and run `pytest` whenever code is pushed or a pull request is opened. Any failing test blocks the merge, turning your test suite into an active quality gate rather than a passive record. For collaborative or long-lived scientific software, CI is one of the most effective safeguards against gradual quality erosion.